In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import IPython.display
import copy

In [2]:
def bin2dec(bin_str):
    dec = 0
    for bit in bin_str:
        dec = dec * 2 + int(bit)
    return dec

def dec2bin(dec_val, pad):
    return format(int(dec_val), '0' + str(pad) + 'b')

In [3]:
def shift_left(tab, n):
    n = n % len(tab)
    return tab[n:] + tab[:n]

In [4]:
def permute(k, perm):
    return ''.join(k[i] for i in perm)

In [5]:
def xor(bin_str1, bin_str2):
    return ''.join('1' if b1 != b2 else '0' for b1, b2 in zip(bin_str1, bin_str2))

In [6]:
PC1 = [56, 48, 40, 32, 24, 16,  8,  0, 57, 49, 41, 33, 25, 17,  9,  1, 58,
       50, 42, 34, 26, 18, 10,  2, 59, 51, 43, 35, 62, 54, 46, 38, 30, 22,
       14,  6, 61, 53, 45, 37, 29, 21, 13,  5, 60, 52, 44, 36, 28, 20, 12,
        4, 27, 19, 11,  3]

shift_table = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]

PC2 = [13, 16, 10, 23,  0,  4,  2, 27, 14,  5, 20,  9, 22, 18, 11,  3,
       25,  7, 15,  6, 26, 19, 12,  1, 40, 51, 30, 36, 46, 54, 29, 39,
       50, 44, 32, 47, 43, 48, 38, 55, 33, 52, 45, 41, 49, 35, 28, 31]

def key_schedule(key):
    key56 = permute(key, PC1)
    lewy, prawy = key56[:28], key56[28:]

    subkeys = []
    for shift in shift_table:
        lewy = shift_left(lewy, shift)
        prawy = shift_left(prawy, shift)

        key48 = permute(lewy + prawy, PC2)
        subkeys.append(key48)
    return subkeys

In [7]:
def Feistel(message, subkeys, F):
    lewy, prawy = message[:32], message[32:]
    for i in range(16):
        lewy = xor(lewy, F(prawy, subkeys[i]))
        if i < 15:
            lewy, prawy = prawy, lewy
    return lewy + prawy

In [8]:
E = [31,  0,  1,  2,  3,  4,  3,  4,  5,  6,  7,  8,  7,  8,  9, 10, 11,
     12, 11, 12, 13, 14, 15, 16, 15, 16, 17, 18, 19, 20, 19, 20, 21, 22,
     23, 24, 23, 24, 25, 26, 27, 28, 27, 28, 29, 30, 31,  0]

SBox = [[[14, 4, 13, 1, 2, 15, 11, 8, 3, 10, 6, 12, 5, 9, 0, 7],
         [ 0, 15, 7, 4, 14, 2, 13, 1, 10, 6, 12, 11, 9, 5, 3, 8],
         [ 4, 1, 14, 8, 13, 6, 2, 11, 15, 12, 9, 7, 3, 10, 5, 0],
         [15, 12, 8, 2, 4, 9, 1, 7, 5, 11, 3, 14, 10, 0, 6, 13]],

        [[15, 1, 8, 14, 6, 11, 3, 4, 9, 7, 2, 13, 12, 0, 5, 10],
         [3, 13, 4, 7, 15, 2, 8, 14, 12, 0, 1, 10, 6, 9, 11, 5],
         [0, 14, 7, 11, 10, 4, 13, 1, 5, 8, 12, 6, 9, 3, 2, 15],
         [13, 8, 10, 1, 3, 15, 4, 2, 11, 6, 7, 12, 0, 5, 14, 9]],

        [[10, 0, 9, 14, 6, 3, 15, 5, 1, 13, 12, 7, 11, 4, 2, 8],
         [13, 7, 0, 9, 3, 4, 6, 10, 2, 8, 5, 14, 12, 11, 15, 1],
         [13, 6, 4, 9, 8, 15, 3, 0, 11, 1, 2, 12, 5, 10, 14, 7],
         [1, 10, 13, 0, 6, 9, 8, 7, 4, 15, 14, 3, 11, 5, 2, 12]],

        [[7, 13, 14, 3, 0, 6, 9, 10, 1, 2, 8, 5, 11, 12, 4, 15],
         [13, 8, 11, 5, 6, 15, 0, 3, 4, 7, 2, 12, 1, 10, 14, 9],
         [10, 6, 9, 0, 12, 11, 7, 13, 15, 1, 3, 14, 5, 2, 8, 4],
         [3, 15, 0, 6, 10, 1, 13, 8, 9, 4, 5, 11, 12, 7, 2, 14]],

        [[2, 12, 4, 1, 7, 10, 11, 6, 8, 5, 3, 15, 13, 0, 14, 9],
         [14, 11, 2, 12, 4, 7, 13, 1, 5, 0, 15, 10, 3, 9, 8, 6],
         [4, 2, 1, 11, 10, 13, 7, 8, 15, 9, 12, 5, 6, 3, 0, 14],
         [11, 8, 12, 7, 1, 14, 2, 13, 6, 15, 0, 9, 10, 4, 5, 3]],

        [[12, 1, 10, 15, 9, 2, 6, 8, 0, 13, 3, 4, 14, 7, 5, 11],
         [10, 15, 4, 2, 7, 12, 9, 5, 6, 1, 13, 14, 0, 11, 3, 8],
         [9, 14, 15, 5, 2, 8, 12, 3, 7, 0, 4, 10, 1, 13, 11, 6],
         [4, 3, 2, 12, 9, 5, 15, 10, 11, 14, 1, 7, 6, 0, 8, 13]],

        [[4, 11, 2, 14, 15, 0, 8, 13, 3, 12, 9, 7, 5, 10, 6, 1],
         [13, 0, 11, 7, 4, 9, 1, 10, 14, 3, 5, 12, 2, 15, 8, 6],
         [1, 4, 11, 13, 12, 3, 7, 14, 10, 15, 6, 8, 0, 5, 9, 2],
         [6, 11, 13, 8, 1, 4, 10, 7, 9, 5, 0, 15, 14, 2, 3, 12]],

        [[13, 2, 8, 4, 6, 15, 11, 1, 10, 9, 3, 14, 5, 0, 12, 7],
         [1, 15, 13, 8, 10, 3, 7, 4, 12, 5, 6, 11, 0, 14, 9, 2],
         [7, 11, 4, 1, 9, 12, 14, 2, 0, 6, 10, 13, 15, 3, 5, 8],
         [2, 1, 14, 7, 4, 10, 8, 13, 15, 12, 9, 0, 3, 5, 6, 11]]]

P = [15,  6, 19, 20, 28, 11, 27, 16,  0, 14, 22, 25,  4, 17, 30,  9,  1,
      7, 23, 13, 31, 26,  2,  8, 18, 12, 29,  5, 21, 10,  3, 24]

def F(right, subkey):
    expanded = permute(right, E)
    xored = xor(expanded, subkey)
    sbox_out = ''
    for i in range(8):
        block = xored[i*6 : (i+1)*6]
        row = bin2dec(block[0] + block[5])
        col = bin2dec(block[1:5])
        val = SBox[i][row][col]
        sbox_out += dec2bin(val, 4)
    return permute(sbox_out, P)

In [9]:
IP = [57, 49, 41, 33, 25, 17,  9,  1, 59, 51, 43, 35, 27, 19, 11,  3, 61,
      53, 45, 37, 29, 21, 13,  5, 63, 55, 47, 39, 31, 23, 15,  7, 56, 48,
      40, 32, 24, 16,  8,  0, 58, 50, 42, 34, 26, 18, 10,  2, 60, 52, 44,
      36, 28, 20, 12,  4, 62, 54, 46, 38, 30, 22, 14,  6]

FP = [39,  7, 47, 15, 55, 23, 63, 31, 38,  6, 46, 14, 54, 22, 62, 30, 37,
       5, 45, 13, 53, 21, 61, 29, 36,  4, 44, 12, 52, 20, 60, 28, 35,  3,
      43, 11, 51, 19, 59, 27, 34,  2, 42, 10, 50, 18, 58, 26, 33,  1, 41,
       9, 49, 17, 57, 25, 32,  0, 40,  8, 48, 16, 56, 24]

def DES(message, subkeys):
    perm_msg = permute(message, IP)
    feistel_out = Feistel(perm_msg, subkeys, F)
    return permute(feistel_out, FP)

# Lista 6 (8 pkt.)

## Zadanie 1 (1 pkt.)

Wczytaj obrazek **miki.png**

In [10]:
img = Image.open('miki.png')
orig_arr = np.array(img)
orig_shape = orig_arr.shape
arr = orig_arr.ravel()
arr_bin = [dec2bin(d, pad=8) for d in arr]
bits = ''.join(arr_bin)
img

i zaszyfruj go za pomocą DESa z kluczem:

In [11]:
key = '0111101000001010110010000001010101111111100000000000101000110001'

stosując szyfrowanie blok po bloku, czyli tak jak na liście 3., tzn. w trybie ECB. Wyświetl zaszyfrowany obrazek.

In [12]:
def split_img(img_bin, n):
    return [img_bin[i:i+n] for i in range(0, len(img_bin), n)]

subkeys = key_schedule(key)

blocks_64 = split_img(bits, 64)
if len(blocks_64[-1]) < 64:
    blocks_64[-1] = blocks_64[-1].ljust(64, '0')

encrypted_blocks = [DES(block, subkeys) for block in blocks_64]
img_t = ''.join(encrypted_blocks)[:len(bits)]

img_ecb_arr = np.array([bin2dec(b) for b in split_img(img_t, 8)], dtype=np.uint8).reshape(orig_shape)
Image.fromarray(img_ecb_arr)

## Zadanie 2 (1 pkt.)

In [13]:
def CBC(msg, key, iv):
    subkeys = key_schedule(key)
    blocks_64 = split_img(msg, 64)
    if len(blocks_64[-1]) < 64:
        blocks_64[-1] = blocks_64[-1].ljust(64, '0')

    encrypted_blocks = []
    prev_szyfr = iv

    for block in blocks_64:
        xored = xor(block, prev_szyfr)
        szyfr_block = DES(xored, subkeys)
        encrypted_blocks.append(szyfr_block)
        prev_szyfr = szyfr_block

    return ''.join(encrypted_blocks)[:len(msg)]

In [14]:
iv = '0' * 64
img_cbc_bits = CBC(bits, key, iv)
img_cbc_arr = np.array([bin2dec(b) for b in split_img(img_cbc_bits, 8)], dtype=np.uint8).reshape(orig_shape)
Image.fromarray(img_cbc_arr)